# RNN full-history aligned benchmark

Этот ноутбук фиксирует командный full-history прогон RNN. В нем RNN-ветка приведена к общей full-history постановке с train/val/test user ids из SimCLR processed split.

Здесь RNN обучается и оценивается на тех же full-history last-512 последовательностях, split и duration метки, которые использовала SimCLR-ветка Алексея. Это нужно только для командного сравнения. Prefix-based ноутбуки 03-11 остаются основной частью RNN-исследования без утечки будущих событий.

## 1. Протокол

- Исходные последовательности: `/Users/fedornikonov/Downloads/курсач/data/processed/*_sequences.pt`.
- Исходные пользователи: `user_ids_train.npy`, `user_ids_val.npy`, `user_ids_test.npy` из того же SimCLR-пакета.
- Метки: `/Users/fedornikonov/Downloads/курсач/data/export/retention_labels.csv`.
- Baseline: `/Users/fedornikonov/Downloads/курсач/data/export/baseline_features.csv`.
- RNN model: GRU, hidden_dim=256, 1 layer, max агрегация, last 512 events.
- Output directory: `artifacts/rnn_full_history_simclr_aligned/`.

Такой прогон не заменяет prefix-based эксперименты. Он нужен для таблицы SimCLR / BERT / RNN в общей презентации.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_DIR = PROJECT_ROOT / "artifacts" / "rnn_full_history_simclr_aligned"
MANIFEST_PATH = OUT_DIR / "manifest.json"

with MANIFEST_PATH.open(encoding="utf-8") as f:
    manifest = json.load(f)

manifest

{'task': 'rnn_full_history_simclr_aligned',
 'simclr_dir': '/Users/fedornikonov/Downloads/курсач',
 'history_mode': 'last_512_events_from_simclr_processed_sequences',
 'embedding_file': 'artifacts/rnn_full_history_simclr_aligned/embeddings/rnn_max_last512_simclr_aligned.parquet',
 'metrics_file': 'artifacts/rnn_full_history_simclr_aligned/aligned_metrics.csv',
 'summary_file': 'artifacts/rnn_full_history_simclr_aligned/aligned_summary.csv',
 'label_summary_file': 'artifacts/rnn_full_history_simclr_aligned/aligned_label_summary.csv',
 'data_summary_file': 'artifacts/rnn_full_history_simclr_aligned/aligned_data_summary.csv',
 'training_history_file': 'artifacts/rnn_full_history_simclr_aligned/aligned_training_history.csv',
 'next_event_metrics_file': 'artifacts/rnn_full_history_simclr_aligned/aligned_next_event_metrics.csv',
 'checkpoint': 'artifacts/rnn_full_history_simclr_aligned/checkpoints/simclr_aligned_gru_h256_l1_last512.pt',
 'pooling': 'max',
 'history_len': 512,
 'seeds': [11, 

## 2. Команда запуска

Полный прогон был выполнен скриптом ниже. Ноутбук читает уже сохраненные артефакты, чтобы не переобучать модель при каждом открытии.

In [2]:
print("""conda run -n ml python scripts/run_rnn_full_history_simclr_aligned.py \
  --out-dir artifacts/rnn_full_history_simclr_aligned \
  --history-len 512 \
  --hidden-dim 256 \
  --num-layers 1 \
  --pooling max \
  --epochs 5 \
  --train-batch-size 64 \
  --embed-batch-size 512 \
  --force-retrain""")

conda run -n ml python scripts/run_rnn_full_history_simclr_aligned.py   --out-dir artifacts/rnn_full_history_simclr_aligned   --history-len 512   --hidden-dim 256   --num-layers 1   --pooling max   --epochs 5   --train-batch-size 64   --embed-batch-size 512   --force-retrain


## 3. Split и метки

Ключевое свойство этого прогона: RNN покрывает полный SimCLR/master split, а не только пересечение с RNN-prepared sequences.

In [3]:
data_summary = pd.read_csv(OUT_DIR / "aligned_data_summary.csv")
label_summary = pd.read_csv(OUT_DIR / "aligned_label_summary.csv")

display(data_summary)
display(label_summary)

,split,users,median_nonpad_len
0,train,47875,118.0
1,val,10259,116.0
2,test,10260,119.0
3,all,68394,118.0


,split,users,retention_1d_positive_rate,retention_7d_positive_rate,retention_14d_positive_rate,retention_30d_positive_rate
0,test,10260,0.251852,0.110819,0.059844,0.003411
1,val,10259,0.250512,0.112779,0.056438,0.003314
2,train,47875,0.249692,0.107050,0.056292,0.003070


In [4]:
coverage = manifest["artifact_contract"]["coverage"]
assert coverage == {"train": 47875, "val": 10259, "test": 10260}, coverage
print("Coverage OK:", coverage)

Coverage OK: {'train': 47875, 'val': 10259, 'test': 10260}


## 4. Качество моделирования последовательности

RNN обучалась без разметки удержания на next-event prediction. Разметка удержания не использовалась при обучении энкодера.

In [5]:
training_history = pd.read_csv(OUT_DIR / "aligned_training_history.csv")
next_event = pd.read_csv(OUT_DIR / "aligned_next_event_metrics.csv")

display(training_history)
display(next_event)

,epoch,train_loss,source
0,1,1.124529,train
1,2,0.777447,train
2,3,0.740306,train
3,4,0.722296,train
4,5,0.710750,train


,split,loss,mrr,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10
0,val,0.804579,0.850148,0.764694,0.957891,0.983234,0.764694,0.957891,0.983234
1,test,0.793147,0.853914,0.771540,0.958577,0.983723,0.771540,0.958577,0.983723


## 5. Оценка удержания

Ниже показаны средние метрики по пяти XGBoost random seeds: 11, 42, 77, 123, 202. Для общего сравнения особенно важны строки `rnn_embedding`: это качество самого RNN-эмбеддинга без ручных baseline-признаки.

In [6]:
summary = pd.read_csv(OUT_DIR / "aligned_summary.csv")
test_summary = summary[summary["split"] == "test"].copy()
cols = ["label", "feature_set", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "pr_auc_std", "num_users"]
display(test_summary[cols].sort_values(["label", "feature_set"]))

,label,feature_set,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,num_users
0,retention_14d,baseline,0.923363,0.000436,0.493949,0.004042,10260
1,retention_14d,baseline_plus_rnn_embedding,0.938778,0.000873,0.556464,0.007619,10260
2,retention_14d,rnn_embedding,0.931938,0.001193,0.496038,0.008942,10260
6,retention_1d,baseline,0.943413,0.000169,0.862641,0.000376,10260
7,retention_1d,baseline_plus_rnn_embedding,0.964081,0.000298,0.900474,0.000954,10260
8,retention_1d,rnn_embedding,0.959816,0.000254,0.881784,0.001152,10260
12,retention_30d,baseline,0.972846,0.001410,0.117670,0.007700,10260
13,retention_30d,baseline_plus_rnn_embedding,0.975745,0.001435,0.238085,0.046530,10260
14,retention_30d,rnn_embedding,0.965843,0.002747,0.190982,0.045624,10260
18,retention_7d,baseline,0.926523,0.000353,0.651112,0.000812,10260


In [7]:
embedding_only = (
    test_summary[test_summary["feature_set"] == "rnn_embedding"]
    [["label", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "pr_auc_std", "num_users"]]
    .sort_values("label")
)
display(embedding_only)

,label,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,num_users
2,retention_14d,0.931938,0.001193,0.496038,0.008942,10260
8,retention_1d,0.959816,0.000254,0.881784,0.001152,10260
14,retention_30d,0.965843,0.002747,0.190982,0.045624,10260
20,retention_7d,0.941303,0.000492,0.677736,0.002734,10260


## 6. Командное сравнение

Текущий RNN-прогон используется для командного full-history сравнения с SimCLR и BERT: во всех строках итоговой таблицы приведена оценка на 10 260 test-пользователях с согласованными retention labels и одинаковыми долями положительного класса. При этом ветки разрабатывались независимо, поэтому таблица не утверждает полного равенства downstream-кода во всех подходах.

In [8]:
team_compare = pd.DataFrame([
    {"horizon": "7d", "model": "BERT mean", "roc_auc": 0.9296, "pr_auc": 0.6228, "test_users": 10260, "positive_rate": "11.1%"},
    {"horizon": "7d", "model": "SimCLR", "roc_auc": 0.9249, "pr_auc": 0.6263, "test_users": 10260, "positive_rate": "11.1%"},
    {"horizon": "7d", "model": "RNN/GRU aligned", "roc_auc": 0.9413, "pr_auc": 0.6777, "test_users": 10260, "positive_rate": "11.1%"},
    {"horizon": "14d", "model": "BERT mean", "roc_auc": 0.9256, "pr_auc": 0.4467, "test_users": 10260, "positive_rate": "6.0%"},
    {"horizon": "14d", "model": "SimCLR", "roc_auc": 0.9189, "pr_auc": 0.4974, "test_users": 10260, "positive_rate": "6.0%"},
    {"horizon": "14d", "model": "RNN/GRU aligned", "roc_auc": 0.9319, "pr_auc": 0.4960, "test_users": 10260, "positive_rate": "6.0%"},
    {"horizon": "30d", "model": "BERT mean", "roc_auc": 0.9665, "pr_auc": 0.1500, "test_users": 10260, "positive_rate": "0.34%"},
    {"horizon": "30d", "model": "SimCLR", "roc_auc": 0.8621, "pr_auc": 0.1725, "test_users": 10260, "positive_rate": "0.34%"},
    {"horizon": "30d", "model": "RNN/GRU aligned", "roc_auc": 0.9658, "pr_auc": 0.1910, "test_users": 10260, "positive_rate": "0.34%"},
])
display(team_compare)

,horizon,model,roc_auc,pr_auc,test_users,positive_rate
0,7d,BERT mean,0.9296,0.6228,10260,11.1%
1,7d,SimCLR,0.9249,0.6263,10260,11.1%
2,7d,RNN/GRU aligned,0.9413,0.6777,10260,11.1%
3,14d,BERT mean,0.9256,0.4467,10260,6.0%
4,14d,SimCLR,0.9189,0.4974,10260,6.0%
5,14d,RNN/GRU aligned,0.9319,0.4960,10260,6.0%
6,30d,BERT mean,0.9665,0.1500,10260,0.34%
7,30d,SimCLR,0.8621,0.1725,10260,0.34%
8,30d,RNN/GRU aligned,0.9658,0.1910,10260,0.34%


Вывод.

Согласованный full-history прогон оценивает RNN на 10 260 test-пользователях, как в общей командной таблице. На SimCLR-compatible labels RNN embedding-only достигает ROC-AUC 0.9413 для retention 7d и 0.9319 для retention 14d. В текущей таблице, выровненной по full-history постановке, тестовой части и retention labels, это выше SimCLR ROC-AUC 0.9249 и 0.9189, а также BERT mean ROC-AUC 0.9296 и 0.9256.

На retention 30d BERT mean и RNN/GRU aligned почти совпадают по ROC-AUC (0.9665 против 0.9658), но RNN выше по PR-AUC (0.1910 против 0.1500). Этот горизонт нужно интерпретировать осторожно, потому что положительный класс составляет только около 0.34% тестовой части.

Итоговая интерпретация: prefix-based RNN остается честной проверкой раннего прогноза без утечки будущих событий, а aligned full-history RNN используется для командного retrospective comparison. Для полностью финального leaderboard остается вынести оценку SimCLR, BERT и RNN-эмбеддинги в один общий downstream evaluator.